# 10 - Prescription - which model, when (and when *not* a GNN)

You've now met the whole family. This capstone does two things: (1) runs a **head-to-head**
across the synthetic suite so the trade-offs are concrete, and (2) distils everything
into a **decision guide** you can actually use.

In [1]:
import os, sys, warnings, time
warnings.filterwarnings("ignore")
sys.path.insert(0, os.path.abspath(".."))   # make utils/ importable from notebooks/

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
pd.set_option("display.precision", 3)
torch.manual_seed(0); np.random.seed(0)

from utils import graphs as G     # synthetic graph generators (known ground truth)
from utils import models as M     # the model zoo (MLP, GCN, SAGE, GAT, GIN, GPS, ...)
from utils import training as T    # train/eval loops + metrics
from utils import plotting as P    # graph drawing, curves, comparisons


## Head-to-head across the suite

Each task below is built to reward a *different* capability. No single model wins
everything - that's the whole point. (`-` marks a model not applicable to that task.)

In [2]:
import numpy as np

def node_clf(data, conv, **kw):
    C = int(data.y.max())+1
    m = (M.MLP(data.num_features,32,C) if conv=="mlp"
         else M.GNN(data.num_features,32,C,conv=conv,**kw))
    T.set_seed(0)
    return T.train_node(m, data, task="classification", epochs=150)["test_metric"]

# Build the tasks once.
sbm, _   = G.make_sbm_homophily(homophily=0.85, seed=1)
het, _   = G.make_heterophilous_graph(homophily=0.2, feature_signal=0.7, seed=1)
role, _  = G.make_structural_role(seed=1)

table = {}
for name, conv in [("MLP","mlp"), ("GCN","gcn"), ("SAGE","sage"), ("GATv2","gatv2")]:
    table[name] = {
        "homophily (SBM)":  node_clf(sbm,  conv),
        "heterophily":      node_clf(het,  conv),
        "structural role":  node_clf(role, conv),
    }
df = pd.DataFrame(table).T
display(df.round(3))

,homophily (SBM),heterophily,structural role
MLP,0.822,0.77,0.385
GCN,0.989,0.97,0.719
SAGE,1.000,0.94,0.677
GATv2,1.000,0.98,0.896


In [3]:
# Graph-level: expressivity (sum vs mean) and long-range (transformer vs MPNN).
expr, _ = G.make_expressivity_graphs(n_graphs=400, seed=0)
lo = T.make_graph_loaders(expr, batch_size=32, seed=0)
g_expr = {}
for name, conv, pool in [("GIN","gin","add"), ("GCN(mean)","gcn","mean")]:
    T.set_seed(0)
    g_expr[name] = T.train_graph(M.GraphGNN(1,32,2,conv=conv,n_layers=3,pool=pool), lo, epochs=40)["test_metric"]

ring, _ = G.make_ring_transfer(num_graphs=400, ring_size=16, num_classes=5, seed=0)
lo = T.make_graph_loaders(ring, batch_size=64, seed=0)
ind = ring[0].num_features
T.set_seed(0); gin = T.train_graph(M.GraphGNN(ind,32,5,conv="gin",n_layers=2,pool="source"), lo, epochs=40)["test_metric"]
T.set_seed(0); gps = T.train_graph(M.GPSModel(ind,32,5,n_layers=2,pool="source"), lo, epochs=40)["test_metric"]

print("Graph-level results")
print(f"  expressivity (density): GIN(sum)={g_expr['GIN']:.3f}  GCN(mean)={g_expr['GCN(mean)']:.3f}")
print(f"  long-range (RingTransfer d=8): GIN={gin:.3f}  GraphGPS={gps:.3f}")

Graph-level results
  expressivity (density): GIN(sum)=0.983  GCN(mean)=0.767
  long-range (RingTransfer d=8): GIN=0.183  GraphGPS=1.000


## What the numbers say

- **Homophilous node classification** -> GCN/SAGE/GAT all strong; GCN is the cheap default.
- **Heterophily** -> GCN sags; **separate ego/neighbour** (SAGE) and heterophily-aware
  models win; the **MLP** is a serious baseline.
- **Structural role** (features are noise) -> models that read topology win; raw features
  are useless - and an MLP is at chance.
- **Expressivity** (graph-level, constant features) -> **GIN (sum)** >> mean aggregation.
- **Long-range** -> **graph transformer (GPS)** >> shallow MPNN.

## The decision guide

```
Is there real RELATIONAL STRUCTURE, and does the target depend on it?
|
+-- NO  (rows are ~independent; signal is in each row's own features)
|        -> DON'T use a GNN. Use a GBDT (XGBoost/LightGBM/CatBoost) or an MLP.
|          (This is the lambda=0 case from notebook 01 - the graph buys nothing.)
|
+-- YES -> what LEVEL is the prediction?
         |
         +-- NODE level
         |     +-- homophilous + good features -> GCN / GraphSAGE   (start here)
         |     +-- neighbours vary in usefulness / want edge weights -> GAT(v2)
         |     +-- huge or growing graph (inductive) -> GraphSAGE + sampling
         |     +-- heterophilous (measure it!) -> H2GCN / GPR-GNN / SAGE; keep an MLP baseline
         |
         +-- EDGE level (recommendation, KG completion)
         |     -> GNN encoder + decoder + negative sampling   (notebook 09)
         |
         +-- GRAPH level (molecules, programs)
               +-- label driven by substructure -> GIN (sum aggr + sum readout)
               +-- long-range dependencies -> graph transformer (GraphGPS / Exphormer)

Going deep and it got WORSE? -> over-smoothing/over-squashing (notebook 06):
   residual / PairNorm / JumpingKnowledge, or rewiring / virtual node / global attention.
```

## When NOT to use a GNN (the honest part)

- **Plain tabular data.** If your rows are independent and the signal is in each row's
  own columns, a **GBDT will usually beat a GNN** and is far simpler - exactly the
  message of the sibling GBDT series. Manufacturing a graph (kNN on features, etc.) often
  adds noise, not signal.
- **No reliable edges.** Garbage structure -> garbage propagation. A GNN is only as good
  as its graph.
- **Strong heterophily with weak structure.** Sometimes the MLP is the right answer.
- **Always run two baselines**: an **MLP on node features** (isolates what structure
  adds) and, for tabular-derived graphs, a **GBDT** (isolates whether you needed a graph
  at all).

## Where to go next

Higher-order & subgraph GNNs (beyond 1-WL), temporal/dynamic graphs, heterogeneous &
knowledge graphs (R-GCN, relational transformers), scalable training (GraphSAINT,
Cluster-GCN), and self-supervised pretraining on graphs. We now have the mental model to
read any of them: **what are its message, aggregate, and update - and which planted
signal would it recover?**